In [1]:
!pip install -q transformers accelerate datasets peft

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [3]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [4]:
from google.colab import files
uploaded = files.upload()

Saving revisedsupportassistant_data.txt to revisedsupportassistant_data.txt


In [5]:
file_name = "revisedsupportassistant_data.txt"  # change this

with open(file_name, "r", encoding="utf-8") as f:
    data = f.read()

In [6]:
from datasets import Dataset

conversations = data.strip().split("\n\n")  # split chats

dataset = Dataset.from_dict({"text": conversations})

In [7]:
from transformers import AutoTokenizer

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded successfully ✅")

Tokenizer loaded successfully ✅


In [8]:
def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(tokenize)

Map:   0%|          | 0/28 [00:00<?, ? examples/s]

In [9]:
!pip install -q peft==0.8.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 7.7 MB/s eta 0:00:00


In [10]:
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)

In [11]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",              # model save hoga yaha
    per_device_train_batch_size=1,       # GPU memory kam hai → 1 rakho
    gradient_accumulation_steps=4,       # effective batch size = 4
    num_train_epochs=3,                  # start small (later increase)
    logging_steps=10,                    # har 10 step pe loss dikhega
    save_steps=50,
    learning_rate=2e-4,
    fp16=False,
    bf16=False,# GPU fast training
    report_to="none"
)

In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

trainer.train()

Step,Training Loss
10,1.210483
20,1.076077


TrainOutput(global_step=21, training_loss=1.1380645831425984, metrics={'train_runtime': 87.5028, 'train_samples_per_second': 0.96, 'train_steps_per_second': 0.24, 'total_flos': 267244516933632.0, 'train_loss': 1.1380645831425984, 'epoch': 3.0})

RAG IMPLEMENTATION

In [1]:
!pip install langchain chromadb sentence-transformers pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 1.9 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Fou

In [2]:
!pip install -q langchain-community
!pip install -q langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-exporter-otlp-proto-common==1.38.0, but you have opentelemetry-exporter-otlp-proto-common 1.41.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-proto==1.38.0, but you have opentelemetry-proto 1.41.0 which is incompatible.


In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [4]:
from google.colab import files
uploaded = files.upload()

Saving Merged_Ecommerce_Guide.pdf to Merged_Ecommerce_Guide.pdf


In [5]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/Merged_Ecommerce_Guide.pdf")
documents = loader.load()

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

docs = text_splitter.split_documents(documents)

In [7]:
!pip install -q langchain-community sentence-transformers

In [8]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [9]:
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

/tmp/ipykernel_10740/4145769993.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
!pip install -U langchain langchain-community

In [11]:
from langchain_community.vectorstores import Chroma

In [12]:
db = Chroma.from_documents(docs, embeddings)

In [13]:
retriever = db.as_retriever()

In [14]:
def rag_response(query):
    docs = retriever.invoke(query)

    # 🔥 FILTER relevant chunks manually
    filtered_docs = []
    for doc in docs:
        if "refund" in doc.page_content.lower():
            filtered_docs.append(doc)

    # fallback if nothing found
    if len(filtered_docs) == 0:
        filtered_docs = docs

    context = "\n\n".join([doc.page_content for doc in filtered_docs])

    prompt = f"""You are a professional e-commerce support assistant.

Answer ONLY using the context below.

Context:
{context}

Question:
{query}

Answer:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        temperature=0.3,
        do_sample=True
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "Answer:" in response:
        response = response.split("Answer:")[-1]

    if "Question:" in response:
        response = response.split("Question:")[0]

    return response.strip()

In [15]:
retriever = db.as_retriever(search_kwargs={"k": 4})

In [41]:
def ft_response(query):
    prompt = f"<|user|>\n{query}\n<|assistant|>"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.7
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


def rag_response(query):
    docs = retriever.invoke(query)
    context = "\n\n".join([doc.page_content for doc in docs])

    # NEW: Natural Response Prompt (No mention of 'context')
    prompt = f"""<|user|>
You are a helpful E-commerce Support Agent. Using your knowledge (provided below), answer the customer directly.

STRICT RULES:
1. Do NOT mention words like "the context", "according to the document", or "based on the text".
2. Start your reply directly (e.g., "Hello! I can help you with that..." or "Yes, you can...").
3. Act as if you are a real person speaking to the customer.

Knowledge:
{context}

Question: {query}
<|assistant|>
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_length = inputs.input_ids.shape[1]

    outputs = model.generate(
        **inputs,
        max_new_tokens=400,
        do_sample=True,
        temperature=0.2, # Accuracy ke liye kam rakha hai
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

    # Cleaning any leftover patterns
    if "Question:" in response:
        response = response.split("Question:")[0]

    return response.strip()




In [42]:
questions = [
    "I live in a remote area. How long does it usually take for delivery?",
    "Can I exchange a product instead of returning it?",
    "I lost the original packaging. Can I still return the product?",
    "I want to merge two accounts. Is that possible",
    "I want to use my loyalty points for a ₹900 order, but it's not working. Why?"

]

In [43]:
compare_models(questions)

Q1: I live in a remote area. How long does it usually take for delivery?

🧠 Fine-tuned Model:
<|user|>
I live in a remote area. How long does it usually take for delivery?
<|assistant|>
It depends on the distance between the delivery location and the nearest transportation hub. Generally, deliveries are made within a few days to a week depending on the distance and the current situation, such as COVID-19 restrictions or other factors. However, it's always a good idea to check with the delivery company or the local authorities for the latest updates on delivery times, especially during crisis situations.

🔎 RAG Model:
The delivery timeline for remote areas may vary depending on the location and product availability. Metro deliveries usually take 2-4 days, while remote areas may take up to 10 days. However, the exact timeline may vary depending on the courier service and logistics sync issues. Customers are advised to check with neighbors or contact support for updates.


Q2: Can I excha